In [1]:
sbox = { "sbox_ascon" : [
0x04, 0x0b, 0x1f, 0x14, 0x1a, 0x15, 0x09, 0x02, 0x1b, 0x05, 0x08, 0x12, 0x1d, 0x03, 0x06, 0x1c,
0x1e, 0x13, 0x07, 0x0e, 0x00, 0x0d, 0x11, 0x18, 0x10, 0x0c, 0x01, 0x19, 0x16, 0x0a, 0x0f, 0x17],
        "sbox_bilgin" : [
0x01, 0x00, 0x19, 0x1a, 0x11, 0x1d, 0x15, 0x1b, 0x14, 0x05, 0x04, 0x17, 0x0e, 0x12, 0x02, 0x1c,
0x0f, 0x08, 0x06, 0x03, 0x0d, 0x07, 0x18, 0x10, 0x1e, 0x09, 0x1f, 0x0a, 0x16, 0x0c, 0x0b, 0x13],
        "sbox_allouzi" : [
0x10, 0x0e, 0x0d, 0x02, 0x0b, 0x11, 0x15, 0x1e, 0x07, 0x18, 0x12, 0x1c, 0x1a, 0x01, 0x0c, 0x06,
0x1f, 0x19, 0x00, 0x17, 0x14, 0x16, 0x08, 0x1b, 0x04, 0x03, 0x13, 0x05, 0x09, 0x0a, 0x1d, 0x0f],
        "sbox_lu_4" : [
0x18, 0x09, 0x1b, 0x06, 0x03, 0x1f, 0x16, 0x01, 0x14, 0x1e, 0x08, 0x05, 0x0a, 0x15, 0x0f, 0x10,
0x04, 0x13, 0x17, 0x0c, 0x1c, 0x00, 0x0d, 0x1a, 0x07, 0x0b, 0x19, 0x12, 0x11, 0x0e, 0x02, 0x1d],
        "sbox_lu_5" : [
0x17, 0x1c, 0x0f, 0x10, 0x02, 0x01, 0x15, 0x1e, 0x19, 0x13, 0x12, 0x0c, 0x0b, 0x08, 0x0d, 0x06,
0x18, 0x0e, 0x00, 0x03, 0x05, 0x1d, 0x0a, 0x1b, 0x04, 0x07, 0x1f, 0x09, 0x1a, 0x16, 0x14, 0x11],
        "sbox_lu_6" : [
0x03, 0x0d, 0x1a, 0x16, 0x11, 0x02, 0x0f, 0x15, 0x00, 0x17, 0x0c, 0x09, 0x14, 0x19, 0x1e, 0x0a,
0x1b, 0x0e, 0x04, 0x1d, 0x1c, 0x08, 0x01, 0x12, 0x07, 0x18, 0x10, 0x13, 0x1f, 0x06, 0x0b, 0x05],
        "sbox_lu_7" : [
0x16, 0x0f, 0x10, 0x09, 0x1b, 0x03, 0x05, 0x06, 0x01, 0x15, 0x1e, 0x12, 0x1c, 0x08, 0x0a, 0x1d,
0x0e, 0x00, 0x0d, 0x1a, 0x18, 0x14, 0x11, 0x1f, 0x13, 0x0c, 0x07, 0x19, 0x0b, 0x17, 0x04, 0x02]
}

sbox_type = ["sbox_ascon",
             "sbox_bilgin",
             "sbox_allouzi",
             "sbox_lu_4",
             "sbox_lu_5",
             "sbox_lu_6",
             "sbox_lu_7",]

num_sbox = len(sbox)

file_lines = []

In [2]:
def sboxNonlinearity(sbox_type):
    """ Calculate a measure of the nonlinearity of an sbox using Walsh Transform.
        Transform is calculated on 255 different 1 bit versions
        of the sbox that are formed by the binary inner product of
        the sbox with sequence 1 through 255.  This is all possible
        linear combinations of the 8 bits into a 1 bit valued sequence
    """
    n = log2n(len(sbox[sbox_type])) # sbox length must be power of 2
    nlv = (2**n-1)*[0] # vector of each nonlinearity calculation
                       # of inner product skipping zero
                       # this just initializes the vector of 255 results

    for c in range(len(nlv)):    # for each of the 255 ways to combine the 8 bits
        t = [ binaryInnerProduct(c+1,sbox[sbox_type][i]) for i in range(len(sbox[sbox_type])) ]
        nlv[c] = nonLinearity(t)
    minNonlinearity = min( [ abs(i) for i in nlv ] )
    maxNonlinearity = max( [ abs(i) for i in nlv ] )
    return minNonlinearity, maxNonlinearity

def walshTransform(t):
    n = log2n(len(t))  # n not used, but asserts if n not a power of 2
    wt = len(t)*[0]
    for w in range( len(t) ):
        for x in range( len(t) ):
            wt[w] = wt[w]+(-1)**(t[x] ^ binaryInnerProduct(w,x) )
    return wt

def binaryInnerProduct(a,b):
    """  """
    ip=0
    ab = a & b
    while ab > 0:
        ip=ip^(ab&1)   # either ^ or + works for walsh transform ...
        ab = ab>>1
    return ip

def nonLinearity(t):
    """ Non-linearity of a binary sequence
    """
    wt = walshTransform(t)
    nl = len(t)/2 - .5*max( [ abs(i) for i in wt ] )
    return nl
    
def log2n(l):
    """ Log2 of an integer only for numbers that are powers of 2 """
    x = l
    n = 0
    while x > 0:
        x=x>>1
        n=n+1
    n = n-1
    assert 2**n == l , "log2n(l) valid only for l=2**n"
    return n
line = "Non-linearity"
print(line)
file_lines.append(line)
minNonlinearity = [0] * num_sbox
maxNonlinearity = [0] * num_sbox
line = "sbox_type       min   max  "
print(line)
file_lines.append(line)
line = "---------------------------"
print(line)
file_lines.append(line)

for i in range(num_sbox):
    minNonlinearity, maxNonlinearity = sboxNonlinearity(sbox_type[i])
    line = sbox_type[i].ljust(16) + str(minNonlinearity).ljust(6) + str(maxNonlinearity).ljust(6)
    print(line)
    file_lines.append(line)
    
line = ""
file_lines.append(line)

Non-linearity
sbox_type       min   max  
---------------------------
sbox_ascon      8.0   12.0  
sbox_bilgin     12.0  12.0  
sbox_allouzi    12.0  12.0  
sbox_lu_4       8.0   12.0  
sbox_lu_5       8.0   12.0  
sbox_lu_6       8.0   12.0  
sbox_lu_7       8.0   12.0  


In [3]:
def differentialUniformity(sbox):
    n = len(sbox)
    max_count = 0

    # Initialize the difference distribution table
    diff_table = [[0 for _ in range(n)] for _ in range(n)]

    for alpha in range(1, n):  # alpha should not be 0
        for x in range(n):
            beta = sbox[x] ^ sbox[x ^ alpha]
            diff_table[alpha][beta] += 1

    # Find the maximum value in the difference distribution table
    for alpha in range(1, n):
        for beta in range(n):
            if diff_table[alpha][beta] > max_count:
                max_count = diff_table[alpha][beta]

    return max_count

In [4]:
line = "Differential Uniformity"
print(line)
file_lines.append(line)
line = "sbox_type       DU"
print(line)
file_lines.append(line)
line = "-------------------"
print(line)
file_lines.append(line)

for i in range(num_sbox):
    delta = differentialUniformity(sbox[sbox_type[i]])
    line = sbox_type[i].ljust(16) + str(delta).ljust(3)
    print(line)
    file_lines.append(line)
    
line = ""
file_lines.append(line)

Differential Uniformity
sbox_type       DU
-------------------
sbox_ascon      8  
sbox_bilgin     2  
sbox_allouzi    2  
sbox_lu_4       8  
sbox_lu_5       8  
sbox_lu_6       8  
sbox_lu_7       6  


**Confusion Coefficient Variance**

In [5]:
import numpy as np

def sbox_dims(sb):
    n = (len(sb) - 1).bit_length()
    assert 1 << n == len(sb), "S-box length must be a power of 2"
    m = max(n, max(sb).bit_length())
    return n, m

def mean(X):
    X = np.asarray(X, dtype=float)
    return np.sum(X, axis=0) / len(X)

def var(X, X_bar):
    X = np.asarray(X, dtype=float)
    return np.sum((X - X_bar) ** 2, axis=0) / len(X)

def CC(sbox, ki, kj, HW=None):
    """Confusion coefficient: E_p[(HW(S(ki^p)) - HW(S(kj^p)))^2]."""
    N = len(sbox)                                  # was hard-coded 256
    if HW is None:
        HW = [bin(v).count("1") for v in range(max(sbox) + 1)]
    hws = []
    for p in range(N):                             # p over F_2^n
        hws.append((HW[sbox[ki ^ p]] - HW[sbox[kj ^ p]]) ** 2)
    return mean(hws)

def CCV(sbox):
    """Variance of the confusion coefficients over all key pairs ki != kj.
       Higher CCV => higher resistance against DPA."""
    N = len(sbox)                                  # was hard-coded 256
    HW = [bin(v).count("1") for v in range(max(sbox) + 1)]
    mean_hws = []
    for ki in range(N):
        for kj in range(N):
            if ki != kj:
                mean_hws.append(CC(sbox, ki, kj, HW))
    return var(mean_hws, mean(mean_hws))

In [6]:
import numpy as np

line = "Confusion Coefficient Variance"
print(line)
file_lines.append(line)
line = "sbox_type       CCV"
print(line)
file_lines.append(line)
line = "---------------------"
print(line)
file_lines.append(line)

for i in range(num_sbox):
    confcoeff = CCV(sbox[sbox_type[i]])
    line = sbox_type[i].ljust(16) + str(round(confcoeff,4)).ljust(5)
    print(line)
    file_lines.append(line)
    
line = ""
file_lines.append(line)

Confusion Coefficient Variance
sbox_type       CCV
---------------------
sbox_ascon      0.5016
sbox_bilgin     0.308
sbox_allouzi    0.4048
sbox_lu_4       0.562
sbox_lu_5       0.2233
sbox_lu_6       0.8887
sbox_lu_7       0.7072


**Minimum Confusion Coefficient**

In [7]:
import numpy as np

def mean(X):
    X = np.asarray(X, dtype=float)
    return np.sum(X, axis=0) / len(X)

def k_prime(sbox, k_star, k, HW=None):
    """kappa'(k*, k) = E_p[ ((HW(S(k*^p)) - HW(S(k^p))) / 2)^2 ]"""
    N = len(sbox)                                   # was hard-coded 256
    if HW is None:
        HW = [bin(v).count("1") for v in range(max(sbox) + 1)]
    hws = []
    for p in range(N):                              # p over F_2^n
        hws.append(((HW[sbox[k_star ^ p]] - HW[sbox[k ^ p]]) / 2) ** 2)
    return mean(hws)

def MCC(sbox):
    """Minimum confusion coefficient  min_{k != k*} kappa'(k*, k).
       Lower MCC => lower DPA/CPA success probability in the low-SNR regime."""
    N = len(sbox)                                   # was hard-coded 256
    HW = [bin(v).count("1") for v in range(max(sbox) + 1)]

    # The distribution of kappa'(k*, k) is independent of the choice of k*
    # (values are only permuted), so a single k* = 0 suffices.
    k_star = 0
    return min(k_prime(sbox, k_star, k, HW) for k in range(N) if k != k_star)

In [8]:
line = "Minimum Confusion Coefficient"
print(line); file_lines.append(line)
line = "sbox_type       MCC"
print(line); file_lines.append(line)
line = "---------------------"
print(line); file_lines.append(line)

for name in sbox_type:
    confcoeff = MCC(sbox[name])
    line = name.ljust(16) + f"{confcoeff:.4f}"
    print(line); file_lines.append(line)

line = ""
print(line); file_lines.append(line)

Minimum Confusion Coefficient
sbox_type       MCC
---------------------
sbox_ascon      0.2500
sbox_bilgin     0.3750
sbox_allouzi    0.3750
sbox_lu_4       0.1562
sbox_lu_5       0.3750
sbox_lu_6       0.2500
sbox_lu_7       0.3438



**Revisited Transparency Order (VTO)**

In [9]:
"""
Revisited Transparency Order (VTO), Li et al. 2020.

    VTO(F) = max_{beta in F_2^m} [ m - 1/(2^{2n}-2^n) *
                 sum_{a in F_2^n \ {0}} | sum_{i=1..m} sum_{j=1..m}
                        (-1)^{beta_i xor beta_j} * C_{F_i,F_j}(a) | ]

with the cross-correlation of two coordinate functions

    C_{F_i,F_j}(a) = sum_{x in F_2^n} (-1)^{F_i(x) xor F_j(x xor a)}
"""

import numpy as np


# ----------------------------------------------------------------------
# helpers
# ----------------------------------------------------------------------
def coordinate_signs(sbox, n, m):
    """signs[x, i] = (-1)^{F_i(x)}   ->  shape (2^n, m)"""
    S = np.asarray(sbox, dtype=np.int64)
    bits = ((S[:, None] >> np.arange(m)[None, :]) & 1)      # F_i(x)
    return 1 - 2 * bits                                     # 0->+1, 1->-1


def cross_correlation_table(sbox, n, m):
    """C[i, j, a] = sum_x (-1)^{F_i(x) xor F_j(x xor a)}"""
    N = 1 << n
    sg = coordinate_signs(sbox, n, m)                       # (N, m)
    C = np.empty((m, m, N), dtype=np.int64)
    x = np.arange(N)
    for a in range(N):
        # (-1)^{u xor v} = (-1)^u * (-1)^v
        C[:, :, a] = sg.T @ sg[x ^ a]                       # (m,m)
    return C



# ----------------------------------------------------------------------
# literal transcription of the VTO formula
# ----------------------------------------------------------------------
def vto(sbox, n, m, beta=None, return_beta=False):
    """
    beta = None  ->  worst case: max over all beta in F_2^m  (the VTO)
    beta = int   ->  the bracket evaluated at that single register state
    """
    denom = 2 ** (2 * n) - 2 ** n                           # 2^{2n} - 2^n
    C = cross_correlation_table(sbox, n, m)                 # C_{F_i,F_j}(a)

    betas = range(1 << m) if beta is None else [beta]       # max_beta  or  fixed beta

    best, arg = -np.inf, None
    for b_val in betas:
        b = (b_val >> np.arange(m)) & 1                     # beta_i, i = 0..m-1
        s = 1 - 2 * b                                       # (-1)^{beta_i}
        W = np.outer(s, s)                                  # (-1)^{beta_i xor beta_j}

        inner = np.einsum('ij,ija->a', W, C)                # double sum over i,j
        total = np.abs(inner[1:]).sum()                     # sum over a != 0, with | . |

        val = m - total / denom
        if val > best:
            best, arg = val, b_val

    return (best, arg) if return_beta else best

In [10]:
from tqdm.notebook import tnrange

line = "ReVisited Transparency Order"
print(line)
file_lines.append(line)
line = "sbox_type       VTO"
print(line)
file_lines.append(line)
line = "----------------------"
print(line)
file_lines.append(line)

for name, sbox_i in sbox.items():
    n = (len(sbox_i) - 1).bit_length()      # e.g. 16 entries -> n = 4
    m = max(n, max(sbox_i).bit_length())    # output width 
    VTO_res = vto(sbox_i, n, m)
    line = name.ljust(16) + f"{VTO_res:.4f}"
    print(line)
    file_lines.append(line)
    
line = ""
file_lines.append(line)

ReVisited Transparency Order
sbox_type       VTO
----------------------
sbox_ascon      4.0000
sbox_bilgin     4.2903
sbox_allouzi    4.1290
sbox_lu_4       4.1290
sbox_lu_5       4.2419
sbox_lu_6       4.2419
sbox_lu_7       4.2419


In [11]:
from tqdm.notebook import tnrange

line = "ReVisited Transparency Order with b=0"
print(line)
file_lines.append(line)
line = "sbox_type       VTO_0"
print(line)
file_lines.append(line)
line = "----------------------"
print(line)
file_lines.append(line)

for name, sbox_i in sbox.items():
    n = (len(sbox_i) - 1).bit_length()      # e.g. 16 entries -> n = 4
    m = max(n, max(sbox_i).bit_length())    # output width 
    VTO_res = vto(sbox_i, n, m, beta=0)
    line = name.ljust(16) + f"{VTO_res:.4f}"
    print(line)
    file_lines.append(line)
    
line = ""
file_lines.append(line)

ReVisited Transparency Order with b=0
sbox_type       VTO_0
----------------------
sbox_ascon      3.9355
sbox_bilgin     4.0968
sbox_allouzi    3.9355
sbox_lu_4       3.8548
sbox_lu_5       4.2419
sbox_lu_6       3.4032
sbox_lu_7       3.5000
